# 策略研发全流程 Pipeline

本 Notebook 是量化策略研发的统一核心入口，严格遵循 `strategy_rules.md` 中的样本内外划分与无前视偏差规则：
- **IS_Train (滚动训练集)**: `2021-01-01` ~ `2023-09-30`，仅用于 Rolling WFV 内部早停与调参。
- **IS_Test (核心测试集)**: `2023-10-01` ~ `2024-09-01`，使用最近 4-Fold 集成进行验证，决定策略是否可以进入实盘预备。
- **OOS (严格样本外)**: `2024-09-01` 之后，作为盲区进行最后一道防线检验。

In [1]:
import sys
import os
import logging
from pathlib import Path

if os.environ.get("OMP_NUM_THREADS") in (None, "0"):
    os.environ["OMP_NUM_THREADS"] = "1"

# 配置项目根目录
ROOT = Path("/root/autodl-tmp/Strategy")
sys.path.insert(0, str(ROOT.parent))

logging.basicConfig(level=logging.INFO, force=True)
for name in ("Strategy", "Strategy.model", "Strategy.backtest"):
    logging.getLogger(name).setLevel(logging.INFO)

from Strategy import config


## 1. 数据准备 (Label & Factors)
在全市场范围内生成所需的 Label 和因子。本 Notebook **训练默认标签 `OPEN930_1000`**：买入价为连续竞价 **首根 K 线**（本仓库分钟数据 `time=931`，与 `OPEN0935` 同理均为真实 bar）→ T+1 **10:00** 卖出（见 `label_generator`）。依赖 **`min_data`** 与 **`Daily_data`**。因子对齐 **T-1 收盘后** 可得。可改用 **`OPEN0935_1000`** 或 **`CLOSE_PRECLOSE`**。


In [ ]:
from Strategy.label.label_generator import generate_and_save_open930_1000_label

LABEL_TAG = "OPEN930_1000"
# 需 min_data 分钟线；Daily_data: CLOSE/PRE_CLOSE（除权）+ LIMIT_UP + LIMIT_DOWN（默认涨跌停剔除）
# 可选: start_date=20210104, end_date=20251231；调试可无涨跌停: use_limit_price_tables=False
open930_label_path = generate_and_save_open930_1000_label()
print(f"[{LABEL_TAG}] Label 已生成: {open930_label_path}")

# 其他示例:
# from Strategy.label.label_generator import generate_and_save_open0935_1000_label, generate_and_save_close_preclose_label
# generate_and_save_open0935_1000_label(save_price_tables=True)
# generate_and_save_close_preclose_label()


In [ ]:
# from Strategy.factor.daily_factor import DailyFactorLibraryAdapter

# # 日频因子需覆盖到当前可用的最新日期
# daily_adapter = DailyFactorLibraryAdapter()
# saved_daily = daily_adapter.compute_and_save_all(
#     start_date=config.IS_TRAIN_START,
#     # end_date=config.IS_TRAIN_END,  # 仅测试时取消注释以加速
# )
# print(f"已保存 {len(saved_daily)} 个日频因子")

In [ ]:
from Strategy.factor.min_factor import MinFactorAdapter
import datetime as dt
# 分钟因子使用 min_data 原始分钟 K 线，取 931-1457 并仅保留 1457 截面
min_adapter = MinFactorAdapter()
saved_min = min_adapter.compute_and_save_all(
    start_date=config.IS_TRAIN_START,
    # end_date=config.IS_TRAIN_END,  # 仅测试时取消注释以加速
    n_workers=12,                  # 可按机器核数调整
    shift_output=True,
)
print(f"已保存 {len(saved_min)} 个分钟因子")

In [ ]:
# from Strategy.factor.factor_base import FactorRegistry
# import Strategy.factor.daily_factor             # 触发纯日频注册因子

# # 计算并保存所有注册的纯日频扩展因子
# FactorRegistry.compute_all()

# # 分钟字段主流程已统一改为 Strategy.factor.min_factor.MinFactorAdapter


## 1.1 GP 因子挖掘
在基础因子 `.fea` 宽表生成后，可选运行 Strategy-native GP 框架，为当前 `LABEL_TAG` 挖掘派生因子。GP 只使用 Train+IS Test 做筛选，OOS 仅保留为报告口径，不参与选择。


In [ ]:
from pathlib import Path

from Strategy.gp_mining.run_gp import mine_gp_factors

# GP 搜索会调用 torch 算子并逐候选做截面评估，正式运行前建议先用小规模参数烟测。
LABEL_TAG = "OPEN930_1000"

RUN_GP_MINING = True
INCLUDE_GP_FACTORS_IN_PANEL = RUN_GP_MINING  # 若要复用已存在的 GP 因子目录，可手动改为 True
GP_EXPERIMENT = f"gp_{LABEL_TAG}_v1"
GP_TERMINAL_NAMES = None  # None 表示使用 outputs/factors/*.fea 全部基础因子；调试可填 ["ma5", "ma10", "rsi_6"]

if RUN_GP_MINING:
    gp_result = mine_gp_factors(
        label_tag=LABEL_TAG,
        experiment=GP_EXPERIMENT,
        generations=32,
        population=64,
        terminal_names=GP_TERMINAL_NAMES,
        save_factors=True,
        overwrite=True,
        device="cuda",
        parent_selection="epsilon_lexicase",
        epsilon_lexicase_epsilon=0.002,
        lexicase_case_sample_ratio=0.35,
    )
    print(f"GP 挖掘完成: evaluated={len(gp_result.leaderboard_rows)}, accepted={len(gp_result.accepted_rows)}")
else:
    print("GP 因子挖掘已跳过。将 RUN_GP_MINING 改为 True 后运行本 cell。")

GP_FACTOR_DIR = Path("/root/autodl-tmp/Strategy/outputs/gp_factors") / GP_EXPERIMENT
print(f"GP 因子目录: {GP_FACTOR_DIR}")


## 2. 面板构建与样本拆分
严格按照配置，物理隔离出 `is_train`, `is_test`, `oos`。模型训练时将绝不碰触 `is_test` 和 `oos`。

In [2]:
from pathlib import Path

import pandas as pd

from Strategy.factor.factor_base import load_all_factors
from Strategy.label.label_generator import load_label
from Strategy.model.trainer import build_panel, split_panel
from Strategy.utils.helpers import ensure_tradedate_as_index

LABEL_TAG = globals().get("LABEL_TAG", "OPEN930_1000")
factor_dict = load_all_factors(dtype="float32")

# # 若上一步运行过 GP 因子挖掘，且 INCLUDE_GP_FACTORS_IN_PANEL=True，则将 GP 因子并入训练特征。
# gp_factor_dir = globals().get("GP_FACTOR_DIR")
# if globals().get("INCLUDE_GP_FACTORS_IN_PANEL", False) and gp_factor_dir is not None:
#     gp_factor_dir = Path(gp_factor_dir)
#     if gp_factor_dir.exists():
#         gp_factor_paths = sorted(gp_factor_dir.glob("*.fea"))
#         for path in gp_factor_paths:
#             factor_dict[f"gp__{path.stem}"] = ensure_tradedate_as_index(pd.read_feather(path))
#         print(f"已加载 {len(gp_factor_paths)} 个 GP 因子: {gp_factor_dir}")

label_df = load_label(LABEL_TAG, dtype="float32")

# 展平构建 Panel
panel = build_panel(factor_dict, label_df)

# 严格切割样本 (依据 config 中定好的常数)
is_train, is_test, oos = split_panel(panel)

print(f"IS_Train: {is_train.shape[0]} rows (截止 {is_train['TRADE_DATE'].max().date()})")
print(f"IS_Test:  {is_test.shape[0]} rows (截止 {is_test['TRADE_DATE'].max().date()})")
print(f"OOS:      {oos.shape[0]} rows (起始 {oos['TRADE_DATE'].min().date()})")

INFO:Strategy.model.trainer:Panel 对齐: 1251 交易日 × 5324 只股票, 226 个因子
INFO:Strategy.model.trainer:正在 concat 227 个 Series ...
INFO:Strategy.model.trainer:Panel 构建完成: shape=(6660324, 229)
INFO:Strategy.model.trainer:panel 日期: [2021-01-18, 2026-03-20]
INFO:Strategy.model.trainer:IS Train: 3497868 rows | IS Test: 1181928 rows | OOS: 1980528 rows


IS_Train: 3497868 rows (截止 2023-09-28)
IS_Test:  1181928 rows (截止 2024-08-30)
OOS:      1980528 rows (起始 2024-09-02)


## 2.1 单因子 IC / 快速分层回测
复用上一步加载的 `factor_dict` 和 `label_df`，对当前这批因子输出与 `single_factor_study/compare_OPEN0935_1000/factor_ic_scan_OPEN0935_1000.csv` 同格式的单因子分析结果。

In [ ]:
from Strategy.backtest.single_factor_research import (
    build_reference_tradeable_mask,
    ic_scan_dataframe,
)

SINGLE_FACTOR_LABEL_TAG = globals().get("LABEL_TAG", "OPEN930_1000")
SINGLE_FACTOR_BT_PRICE_TAG = SINGLE_FACTOR_LABEL_TAG
SINGLE_FACTOR_OUT = config.BT_RESULT_DIR / "single_factor_study" / f"compare_{SINGLE_FACTOR_LABEL_TAG}"
SINGLE_FACTOR_OUT.mkdir(parents=True, exist_ok=True)

RUN_SINGLE_FACTOR_IC_SCAN = True

single_factor_label = label_df if SINGLE_FACTOR_LABEL_TAG == LABEL_TAG else load_label(SINGLE_FACTOR_LABEL_TAG)
single_factor_factors = factor_dict

# 仅使用全区间参考 score 建 IC mask；不要用 *_is_test 文件，否则 Train/OOS 分段会被空掉。
single_factor_mask = None
ref_path = config.SCORE_OUTPUT_DIR / "pipeline_holdout" / f"SCORE_xgb_{SINGLE_FACTOR_LABEL_TAG}.fea"
if ref_path.exists():
    ref_score = pd.read_feather(ref_path).set_index("TRADE_DATE")
    single_factor_mask = build_reference_tradeable_mask(ref_score)
    print("使用参考打分建 IC mask:", ref_path, single_factor_mask.shape)
else:
    print("未找到全区间参考打分, IC scan 使用 mask=None:", ref_path)

scan_path = SINGLE_FACTOR_OUT / f"factor_ic_scan_{SINGLE_FACTOR_LABEL_TAG}.csv"
if RUN_SINGLE_FACTOR_IC_SCAN:
    single_factor_ic_tab = ic_scan_dataframe(
        single_factor_label,
        single_factor_factors,
        tradeable_mask=single_factor_mask,
        min_stocks=10,
    )
    single_factor_ic_tab.to_csv(scan_path, index=False)
    print("已保存:", scan_path, "shape=", single_factor_ic_tab.shape)
else:
    single_factor_ic_tab = pd.read_csv(scan_path)
    print("已加载:", scan_path, "shape=", single_factor_ic_tab.shape)

rank_col = "ic_OOS_mean" if "ic_OOS_mean" in single_factor_ic_tab.columns else "ic_Overall_mean"
if rank_col in single_factor_ic_tab.columns:
    display_tab = single_factor_ic_tab.sort_values(rank_col, ascending=False)
else:
    display_tab = single_factor_ic_tab
print(display_tab.head(20).to_string())


In [ ]:
from Strategy.backtest.single_factor_research import export_factor_report_discrete_ic_then_bt

RUN_SINGLE_FACTOR_REPORTS = True
N_SINGLE_FACTOR_PLOT = 5
SINGLE_FACTOR_EXTRA = []  # 可手动加入指定因子名，如 ["Klineshape_gap_ratio_min"]
TOP_KS = (20, 50, 100)
TAIL_KS = (20, 50, 100)

if RUN_SINGLE_FACTOR_REPORTS:
    rank_col = "ic_OOS_mean" if "ic_OOS_mean" in single_factor_ic_tab.columns else "ic_Overall_mean"
    ranked_tab = single_factor_ic_tab.sort_values(rank_col, ascending=False) if rank_col in single_factor_ic_tab.columns else single_factor_ic_tab
    names = list(ranked_tab.head(N_SINGLE_FACTOR_PLOT)["factor"])
    for extra_name in SINGLE_FACTOR_EXTRA:
        if extra_name in single_factor_factors and extra_name not in names:
            names.append(extra_name)

    for name in names:
        print("---", name, "---")
        export_factor_report_discrete_ic_then_bt(
            name,
            single_factor_factors[name],
            single_factor_label,
            SINGLE_FACTOR_OUT,
            tradeable_mask=single_factor_mask,
            val_start=config.IS_TEST_START,
            end_date=None,
            top_ks=TOP_KS,
            tail_ks=TAIL_KS,
            twap_tag=SINGLE_FACTOR_BT_PRICE_TAG,
        )
print("单因子输出根目录:", SINGLE_FACTOR_OUT)


## 3. IS Train 滚动训练 (Rolling CV)
利用 `is_train` 内部通过时间窗滚动的方式进行训练和早停，并保存模型参数。
与 `strategy_rules.md` 一致：**截面模型**、**`RollingTrainer` 时间块 CV**、神经网络 **`batch_size=1`**（每个优化步仅一个交易日截面；见 `config.NN_TRAINER_BATCH_SIZE`）。


In [3]:
from Strategy.model.rolling_trainer import RollingTrainer
from Strategy.model.trainer import XGBTrainer

rt_xgb = RollingTrainer(
    model_class=XGBTrainer,
    model_kwargs={
        "use_wpcc": True,
    }
)

# 仅在 IS_Train 上执行所有 Fold 的滚动训练
rt_xgb.train_all_folds(is_train)

print("\n========== XGB CV Fold IC ==========")
print(rt_xgb.fold_ic_report())

rt_xgb.save_model(config.SCORE_OUTPUT_DIR / f"rolling_xgb_{LABEL_TAG}.pkl")


INFO:Strategy.model.rolling_trainer:generate_folds: IS Train=[2021-01-01, 2023-09-30], val_months=3 → 11 Fold
INFO:Strategy.model.rolling_trainer:Rolling Val CV: 11 Fold, 特征数=226, IS Train=2021-01-01~2023-09-30
INFO:Strategy.model.rolling_trainer:━━ Fold 1/11: Val=[2021-01-01, 2021-03-31]  Train=IS Train-Val ━━
INFO:Strategy.model.rolling_trainer:  Train: 3242316 行 (609 日) | Val: 255552 行 (48 日)
INFO:Strategy.model.trainer:特征数=226
INFO:Strategy.model.trainer:训练样本 n=2855299; label std=0.0350967; 特征 NaN=0.13%
INFO:Strategy.model.trainer:使用 WPCC 自定义目标函数


[0]	train-rmse:0.50043	train-wpcc:-0.03827	val-rmse:0.50018	val-wpcc:-0.05257
[85]	train-rmse:0.50043	train-wpcc:-0.14143	val-rmse:0.50019	val-wpcc:-0.07114


INFO:Strategy.model.trainer:训练完成. best_iteration=5
INFO:Strategy.model.rolling_trainer:  Fold 1 Val  IC=0.0711  ICIR=1.1418  RankIC=0.0680  days=48
INFO:Strategy.model.rolling_trainer:━━ Fold 2/11: Val=[2021-04-01, 2021-06-30]  Train=IS Train-Val ━━
INFO:Strategy.model.rolling_trainer:  Train: 3178428 行 (597 日) | Val: 319440 行 (60 日)
INFO:Strategy.model.trainer:特征数=226
INFO:Strategy.model.trainer:训练样本 n=2799039; label std=0.0353843; 特征 NaN=0.13%
INFO:Strategy.model.trainer:使用 WPCC 自定义目标函数


[0]	train-rmse:0.50057	train-wpcc:-0.04178	val-rmse:0.49878	val-wpcc:-0.04630
[100]	train-rmse:0.50057	train-wpcc:-0.14627	val-rmse:0.49878	val-wpcc:-0.06489
[108]	train-rmse:0.50057	train-wpcc:-0.14947	val-rmse:0.49878	val-wpcc:-0.06102


INFO:Strategy.model.trainer:训练完成. best_iteration=28
INFO:Strategy.model.rolling_trainer:  Fold 2 Val  IC=0.0610  ICIR=1.1776  RankIC=0.0445  days=60
INFO:Strategy.model.rolling_trainer:━━ Fold 3/11: Val=[2021-07-01, 2021-09-30]  Train=IS Train-Val ━━
INFO:Strategy.model.rolling_trainer:  Train: 3157132 行 (593 日) | Val: 340736 行 (64 日)
INFO:Strategy.model.trainer:特征数=226
INFO:Strategy.model.trainer:训练样本 n=2772423; label std=0.034786; 特征 NaN=0.13%
INFO:Strategy.model.trainer:使用 WPCC 自定义目标函数


...

INFO:Strategy.model.trainer:训练完成. best_iteration=61
INFO:Strategy.model.rolling_trainer:  Fold 10 Val  IC=0.0216  ICIR=0.4574  RankIC=0.0248  days=59
INFO:Strategy.model.rolling_trainer:━━ Fold 11/11: Val=[2023-07-01, 2023-09-30]  Train=IS Train-Val ━━
INFO:Strategy.model.rolling_trainer:  Train: 3157132 行 (593 日) | Val: 340736 行 (64 日)
INFO:Strategy.model.trainer:特征数=226
INFO:Strategy.model.trainer:训练样本 n=2732798; label std=0.0359459; 特征 NaN=0.14%
INFO:Strategy.model.trainer:使用 WPCC 自定义目标函数


[0]	train-rmse:0.50023	train-wpcc:-0.04328	val-rmse:0.50197	val-wpcc:-0.02564
[87]	train-rmse:0.50024	train-wpcc:-0.14393	val-rmse:0.50197	val-wpcc:-0.03083


INFO:Strategy.model.trainer:训练完成. best_iteration=7
INFO:Strategy.model.rolling_trainer:  Fold 11 Val  IC=0.0308  ICIR=0.6632  RankIC=0.0219  days=64
INFO:Strategy.model.rolling_trainer:Rolling Val CV 完成: 11 Fold 已训练
INFO:Strategy.model.rolling_trainer:滚动模型已保存: /root/autodl-tmp/Strategy/outputs/scores/rolling_xgb_OPEN930_1000.pkl (11 folds)



========== XGB CV Fold IC ==========
    fold_id status  val_ic_mean  val_ic_std  val_icir  val_rank_ic_mean  \
0         1     ok     0.071138    0.062302  1.141832          0.068041   
1         2     ok     0.061022    0.051819  1.177600          0.044484   
2         3     ok     0.045376    0.046824  0.969086          0.066772   
3         4     ok     0.038307    0.051264  0.747243          0.028995   
4         5     ok     0.055542    0.046991  1.181956          0.052473   
5         6     ok     0.041886    0.049009  0.854671          0.036254   
6         7     ok     0.034872    0.043876  0.794777          0.031857   
7         8     ok     0.045822    0.046578  0.983760          0.049737   
8         9     ok     0.037928    0.058717  0.645947          0.034511   
9        10     ok     0.021630    0.047289  0.457394          0.024769   
10       11     ok     0.030830    0.046488  0.663171          0.021928   

    n_val_days  
0           48  
1           60  
2         

PosixPath('/root/autodl-tmp/Strategy/outputs/scores/rolling_xgb_OPEN930_1000.pkl')

In [4]:
from Strategy.model.transformer_trainer import TransformerTrainer

rt_tf = RollingTrainer(
    model_class=TransformerTrainer,
    model_kwargs={
        "d_model": 64, "nhead": 4, "num_layers": 2, "d_ff": 128, "dropout": 0.15,
        "epochs": 50, "lr": 5e-4, "weight_decay": 0.01, "batch_size": 1,
        "early_stopping_patience": 8, "device": "cuda", "use_amp": False,
    }
)

rt_tf.train_all_folds(is_train)

print("\n========== Transformer CV Fold IC ==========")
print(rt_tf.fold_ic_report())
rt_tf.save_model(config.SCORE_OUTPUT_DIR / f"rolling_transformer_{LABEL_TAG}.pkl")


INFO:Strategy.model.rolling_trainer:generate_folds: IS Train=[2021-01-01, 2023-09-30], val_months=3 → 11 Fold
INFO:Strategy.model.rolling_trainer:Rolling Val CV: 11 Fold, 特征数=226, IS Train=2021-01-01~2023-09-30
INFO:Strategy.model.rolling_trainer:━━ Fold 1/11: Val=[2021-01-01, 2021-03-31]  Train=IS Train-Val ━━
INFO:Strategy.model.rolling_trainer:  Train: 3242316 行 (609 日) | Val: 255552 行 (48 日)
INFO:Strategy.model.transformer_trainer:训练设备: cuda, AMP: False
INFO:Strategy.model.transformer_trainer:CrossSectionalTransformer: d_input=226 d_model=64 nhead=4 layers=2 params=0.08M
INFO:Strategy.model.transformer_trainer:训练集: 609 日 | 验证集: 48 日 | 特征: 226
INFO:Strategy.model.transformer_trainer:  Epoch 1/50  train=-0.00042  val=-0.00669  lr=5.00e-04
INFO:Strategy.model.transformer_trainer:  Epoch 2/50  train=-0.00175  val=-0.01259  lr=4.98e-04
INFO:Strategy.model.transformer_trainer:  Epoch 3/50  train=-0.00421  val=-0.00986  lr=4.96e-04
INFO:Strategy.model.transformer_trainer:  Epoch 5/50  tra


========== Transformer CV Fold IC ==========
    fold_id status  val_ic_mean  val_ic_std  val_icir  val_rank_ic_mean  \
0         1     ok     0.017631    0.073172  0.240959          0.018163   
1         2     ok     0.015206    0.041867  0.363187          0.017897   
2         3     ok    -0.005832    0.074434 -0.078346          0.009822   
3         4     ok     0.004886    0.075807  0.064454         -0.001689   
4         5     ok     0.014644    0.044863  0.326408          0.020925   
5         6     ok     0.025889    0.067622  0.382844          0.015110   
6         7     ok     0.009765    0.070744  0.138033         -0.007844   
7         8     ok     0.016642    0.062931  0.264450          0.021374   
8         9     ok     0.016352    0.055885  0.292604          0.019869   
9        10     ok     0.005279    0.086087  0.061323          0.015684   
10       11     ok    -0.000712    0.073786 -0.009653         -0.015461   

    n_val_days  
0           48  
1           60  
2 

PosixPath('/root/autodl-tmp/Strategy/outputs/scores/rolling_transformer_OPEN930_1000.pkl')

In [5]:
from Strategy.model.mlp_trainer import MLPTrainer

rt_mlp = RollingTrainer(
    model_class=MLPTrainer,
    model_kwargs={
        "hidden_dims": (256, 128, 64),
        "dropout": 0.15,
        "epochs": 50,
        "lr": 5e-4,
        "weight_decay": 0.01,
        "batch_size": 1,
        "early_stopping_patience": 8,
        "device": "cuda",
        "use_amp": False,
    },
)

rt_mlp.train_all_folds(is_train)

print("\n========== MLP CV Fold IC ==========")
print(rt_mlp.fold_ic_report())
rt_mlp.save_model(config.SCORE_OUTPUT_DIR / f"rolling_mlp_{LABEL_TAG}.pkl")


INFO:Strategy.model.rolling_trainer:generate_folds: IS Train=[2021-01-01, 2023-09-30], val_months=3 → 11 Fold
INFO:Strategy.model.rolling_trainer:Rolling Val CV: 11 Fold, 特征数=226, IS Train=2021-01-01~2023-09-30
INFO:Strategy.model.rolling_trainer:━━ Fold 1/11: Val=[2021-01-01, 2021-03-31]  Train=IS Train-Val ━━
INFO:Strategy.model.rolling_trainer:  Train: 3242316 行 (609 日) | Val: 255552 行 (48 日)
INFO:Strategy.model.mlp_trainer:MLP 训练设备: cuda, AMP: False
INFO:Strategy.model.mlp_trainer:CrossSectionalMLP: d_input=226 hidden=(256, 128, 64) params=0.10M
INFO:Strategy.model.mlp_trainer:训练集: 609 日 | 验证集: 48 日 | 特征: 226
INFO:Strategy.model.mlp_trainer:  Epoch 1/50  train=-0.00212  val=-0.01339  lr=5.00e-04
INFO:Strategy.model.mlp_trainer:  Epoch 2/50  train=-0.00252  val=-0.01139  lr=4.98e-04
INFO:Strategy.model.mlp_trainer:  Epoch 3/50  train=-0.00491  val=-0.00724  lr=4.96e-04
INFO:Strategy.model.mlp_trainer:  Epoch 5/50  train=-0.00541  val=-0.01018  lr=4.88e-04
INFO:Strategy.model.mlp_tra


========== MLP CV Fold IC ==========
    fold_id status  val_ic_mean  val_ic_std  val_icir  val_rank_ic_mean  \
0         1     ok     0.012782    0.075341  0.169654          0.009399   
1         2     ok     0.016181    0.032106  0.503988          0.015127   
2         3     ok     0.022295    0.089663  0.248659          0.018065   
3         4     ok     0.007455    0.058506  0.127423          0.000237   
4         5     ok     0.000501    0.043240  0.011590         -0.001248   
5         6     ok     0.022604    0.067660  0.334084          0.012346   
6         7     ok     0.011232    0.071789  0.156457          0.007503   
7         8     ok     0.014582    0.069854  0.208752          0.012494   
8         9     ok     0.010295    0.056069  0.183608          0.012306   
9        10     ok    -0.017508    0.055475 -0.315605         -0.042208   
10       11     ok    -0.008754    0.056698 -0.154393         -0.016702   

    n_val_days  
0           48  
1           60  
2         

PosixPath('/root/autodl-tmp/Strategy/outputs/scores/rolling_mlp_OPEN930_1000.pkl')

## 4. IS Test 核心推理与模型集成
利用最近的 4-Fold 模型对 `is_test` 进行打分，随后进行跨模型种类（XGB / Transformer / MLP）的集成。


In [6]:
from Strategy.model.scorer import mask_scores_by_price
from Strategy.data_io.saver import save_wide_table
from Strategy.model.ensemble_scorer import compute_score_correlation, select_ensemble_models, ensemble_scores

score_test_xgb = rt_xgb.predict_is_test(is_test, normalize=True)
score_test_xgb = mask_scores_by_price(score_test_xgb, label_tag=LABEL_TAG)
save_wide_table(score_test_xgb, config.SCORE_OUTPUT_DIR / f"SCORE_xgb_{LABEL_TAG}_is_test.fea")

score_test_tf = rt_tf.predict_is_test(is_test, normalize=True)
score_test_tf = mask_scores_by_price(score_test_tf, label_tag=LABEL_TAG)
save_wide_table(score_test_tf, config.SCORE_OUTPUT_DIR / f"SCORE_transformer_{LABEL_TAG}_is_test.fea")

score_test_mlp = rt_mlp.predict_is_test(is_test, normalize=True)
score_test_mlp = mask_scores_by_price(score_test_mlp, label_tag=LABEL_TAG)
save_wide_table(score_test_mlp, config.SCORE_OUTPUT_DIR / f"SCORE_mlp_{LABEL_TAG}_is_test.fea")

score_dfs = {
    "xgb": score_test_xgb,
    "transformer": score_test_tf,
    "mlp": score_test_mlp,
}

corr = compute_score_correlation(score_dfs)
print("\n模型打分截面相关性矩阵:")
print(corr)

selected = select_ensemble_models(score_dfs, max_pairwise_corr=0.9)

score_test_ens = ensemble_scores(
    score_dfs,
    selected_models=selected,
    label_tag=f"{LABEL_TAG}_is_test",
    save=True,
    output_dir=config.SCORE_OUTPUT_DIR
)


INFO:Strategy.model.rolling_trainer:IS Test Ensemble: 选取最近 4 个 Fold → Fold IDs=[11, 10, 9, 8]  Val Ends=[datetime.date(2023, 9, 30), datetime.date(2023, 6, 30), datetime.date(2023, 3, 31), datetime.date(2022, 12, 31)]
INFO:Strategy.model.rolling_trainer:  Fold 11 推理完成: mean=0.5001 std=0.0003
INFO:Strategy.model.rolling_trainer:  Fold 10 推理完成: mean=0.5001 std=0.0004
INFO:Strategy.model.rolling_trainer:  Fold 9 推理完成: mean=0.5001 std=0.0003
INFO:Strategy.model.rolling_trainer:  Fold 8 推理完成: mean=0.5001 std=0.0005
INFO:Strategy.model.rolling_trainer:IS Test Ensemble 推理完成: 222 dates × 5324 stocks
INFO:Strategy.model.scorer:price mask: before=1181928 after=1130010 removed=51918
INFO:Strategy.model.rolling_trainer:IS Test Ensemble: 选取最近 4 个 Fold → Fold IDs=[11, 10, 9, 8]  Val Ends=[datetime.date(2023, 9, 30), datetime.date(2023, 6, 30), datetime.date(2023, 3, 31), datetime.date(2022, 12, 31)]
INFO:Strategy.model.rolling_trainer:  Fold 11 推理完成: mean=-1.1736 std=1.6870
INFO:Strategy.model.rolli


模型打分截面相关性矩阵:
                  mlp  transformer       xgb
mlp          1.000000    -0.107160  0.063113
transformer -0.107160     1.000000  0.108552
xgb          0.063113     0.108552  1.000000


INFO:Strategy.model.ensemble_scorer:模型打分相关性矩阵 (222 日均):
               mlp  transformer    xgb
mlp          1.000       -0.107  0.063
transformer -0.107        1.000  0.109
xgb          0.063        0.109  1.000
INFO:Strategy.model.ensemble_scorer:跨模型集成入选: ['xgb', 'transformer', 'mlp'] (3 / 3)
INFO:Strategy.model.ensemble_scorer:跨模型集成: 3 个模型 → ['xgb', 'transformer', 'mlp']
/root/autodl-tmp/Strategy/model/ensemble_scorer.py:256: RuntimeWarning: Mean of empty slice
  avg_scores = np.nanmean(stacked, axis=0)                       # (dates, stocks)
INFO:Strategy.model.ensemble_scorer:跨模型集成打分: 222 dates × 5324 stocks
INFO:Strategy.model.ensemble_scorer:跨模型集成打分已保存: /root/autodl-tmp/Strategy/outputs/scores/SCORE_ensemble_OPEN930_1000_is_test.fea


## 5. 快速业绩回测 (仅观测 IS_Test)
在没有发生前视偏差的数据区间上观测策略真实的盈利能力。

In [7]:
import sys
import logging
from pathlib import Path

ROOT = Path("/root/autodl-tmp/Strategy")
if str(ROOT.parent) not in sys.path:
    sys.path.insert(0, str(ROOT.parent))
logging.basicConfig(level=logging.INFO, force=True)

from Strategy import config
from Strategy.label.label_generator import load_label
from Strategy.model.scorer import load_is_test_scores_from_disk
from Strategy.backtest.quick_backtest import run_quick_backtest

LABEL_TAG = "OPEN930_1000"

label_df = load_label(LABEL_TAG)
models_to_test = load_is_test_scores_from_disk(LABEL_TAG)

for name, score_df in models_to_test.items():
    print(f"\n{'='*50}\n回测评估: {name} (IS_Test)\n{'='*50}")
    run_quick_backtest(
        score_df=score_df,
        label_df=label_df,
        n_groups=20,
        output_dir=config.BT_RESULT_DIR / "is_test" / name,
        start_date=config.IS_TEST_START,
        top_ks=(20, 50, 100),
        tail_ks=(20, 50, 100),
        run_ic=True,
        title=f"{name.upper()} | {LABEL_TAG} | IS_Test",
    )


INFO:Strategy.model.scorer:price mask: before=1130010 after=1130010 removed=0
INFO:Strategy.model.scorer:price mask: before=1130010 after=1130010 removed=0
INFO:Strategy.model.scorer:price mask: before=1130010 after=1130010 removed=0
INFO:Strategy.model.scorer:price mask: before=1130010 after=1130010 removed=0



回测评估: xgb (IS_Test)


INFO:Strategy.backtest.quick_backtest:分层回测完成: 222 交易日, 20 组, topK=(20, 50, 100), tailK=(20, 50, 100)
INFO:Strategy.backtest.quick_backtest:stock pool: score=5090 tradeable=3009 no_twap=0 limit_up=32 new=6 delist=0 st=564 prefix=1477
INFO:Strategy.backtest.quick_backtest:  group1: Ann=-34.07% MDD=-51.85% SR=-0.74 WR=46.85%
INFO:Strategy.backtest.quick_backtest:  group2: Ann=-9.14% MDD=-46.26% SR=-0.21 WR=49.55%
INFO:Strategy.backtest.quick_backtest:  group3: Ann=-10.31% MDD=-44.76% SR=-0.25 WR=50.45%
INFO:Strategy.backtest.quick_backtest:  group4: Ann=-13.72% MDD=-44.00% SR=-0.34 WR=49.10%
INFO:Strategy.backtest.quick_backtest:  group5: Ann=-10.86% MDD=-43.79% SR=-0.29 WR=49.10%
INFO:Strategy.backtest.quick_backtest:  group6: Ann=-6.05% MDD=-41.54% SR=-0.16 WR=50.90%
INFO:Strategy.backtest.quick_backtest:  group7: Ann=-8.64% MDD=-41.78% SR=-0.24 WR=50.45%
INFO:Strategy.backtest.quick_backtest:  group8: Ann=-14.45% MDD=-41.41% SR=-0.43 WR=49.10%
INFO:Strategy.backtest.quick_backtest:  gr


回测评估: transformer (IS_Test)


INFO:Strategy.backtest.quick_backtest:分层回测完成: 222 交易日, 20 组, topK=(20, 50, 100), tailK=(20, 50, 100)
INFO:Strategy.backtest.quick_backtest:stock pool: score=5090 tradeable=3009 no_twap=0 limit_up=32 new=6 delist=0 st=564 prefix=1477
INFO:Strategy.backtest.quick_backtest:  group1: Ann=-31.41% MDD=-48.00% SR=-0.82 WR=45.05%
INFO:Strategy.backtest.quick_backtest:  group2: Ann=-25.62% MDD=-45.75% SR=-0.69 WR=45.50%
INFO:Strategy.backtest.quick_backtest:  group3: Ann=-28.11% MDD=-45.15% SR=-0.79 WR=48.20%
INFO:Strategy.backtest.quick_backtest:  group4: Ann=-18.46% MDD=-43.86% SR=-0.51 WR=46.40%
INFO:Strategy.backtest.quick_backtest:  group5: Ann=-25.87% MDD=-44.30% SR=-0.73 WR=44.14%
INFO:Strategy.backtest.quick_backtest:  group6: Ann=-22.08% MDD=-44.42% SR=-0.61 WR=44.59%
INFO:Strategy.backtest.quick_backtest:  group7: Ann=-21.58% MDD=-43.51% SR=-0.63 WR=46.40%
INFO:Strategy.backtest.quick_backtest:  group8: Ann=-20.21% MDD=-42.97% SR=-0.58 WR=49.10%
INFO:Strategy.backtest.quick_backtest: 


回测评估: mlp (IS_Test)


INFO:Strategy.backtest.quick_backtest:分层回测完成: 222 交易日, 20 组, topK=(20, 50, 100), tailK=(20, 50, 100)
INFO:Strategy.backtest.quick_backtest:stock pool: score=5090 tradeable=3009 no_twap=0 limit_up=32 new=6 delist=0 st=564 prefix=1477
INFO:Strategy.backtest.quick_backtest:  group1: Ann=-26.02% MDD=-30.36% SR=-0.91 WR=46.85%
INFO:Strategy.backtest.quick_backtest:  group2: Ann=-19.74% MDD=-30.73% SR=-0.73 WR=48.20%
INFO:Strategy.backtest.quick_backtest:  group3: Ann=-20.28% MDD=-31.36% SR=-0.73 WR=46.85%
INFO:Strategy.backtest.quick_backtest:  group4: Ann=-16.90% MDD=-31.46% SR=-0.60 WR=46.85%
INFO:Strategy.backtest.quick_backtest:  group5: Ann=-12.99% MDD=-32.18% SR=-0.44 WR=46.85%
INFO:Strategy.backtest.quick_backtest:  group6: Ann=-22.40% MDD=-36.39% SR=-0.76 WR=46.40%
INFO:Strategy.backtest.quick_backtest:  group7: Ann=-24.06% MDD=-38.49% SR=-0.78 WR=48.20%
INFO:Strategy.backtest.quick_backtest:  group8: Ann=-19.06% MDD=-39.62% SR=-0.60 WR=46.40%
INFO:Strategy.backtest.quick_backtest: 


回测评估: ensemble (IS_Test)


INFO:Strategy.backtest.quick_backtest:分层回测完成: 222 交易日, 20 组, topK=(20, 50, 100), tailK=(20, 50, 100)
INFO:Strategy.backtest.quick_backtest:stock pool: score=5090 tradeable=3009 no_twap=0 limit_up=32 new=6 delist=0 st=564 prefix=1477
INFO:Strategy.backtest.quick_backtest:  group1: Ann=-20.88% MDD=-48.96% SR=-0.48 WR=47.30%
INFO:Strategy.backtest.quick_backtest:  group2: Ann=-18.14% MDD=-42.55% SR=-0.45 WR=48.65%
INFO:Strategy.backtest.quick_backtest:  group3: Ann=-23.93% MDD=-42.37% SR=-0.65 WR=47.75%
INFO:Strategy.backtest.quick_backtest:  group4: Ann=-21.04% MDD=-42.18% SR=-0.58 WR=46.85%
INFO:Strategy.backtest.quick_backtest:  group5: Ann=-17.83% MDD=-40.24% SR=-0.52 WR=47.75%
INFO:Strategy.backtest.quick_backtest:  group6: Ann=-18.67% MDD=-40.44% SR=-0.55 WR=48.20%
INFO:Strategy.backtest.quick_backtest:  group7: Ann=-12.15% MDD=-38.60% SR=-0.37 WR=49.55%
INFO:Strategy.backtest.quick_backtest:  group8: Ann=-17.51% MDD=-40.62% SR=-0.53 WR=46.40%
INFO:Strategy.backtest.quick_backtest: 

## 6. OOS 盲区验证 (严格门控)
⚠️ **重要提示**：此部分代码仅当上方 `IS_Test` 回测结果各项指标完全达到预期目标，决定实盘部署时，才允许运行！一旦针对 OOS 数据调整超参，OOS 就被“污染”了。

In [ ]:
# score_oos_xgb = rt_xgb.predict_is_test(oos, normalize=True)
# score_oos_tf = rt_tf.predict_is_test(oos, normalize=True)
# score_oos_mlp = rt_mlp.predict_is_test(oos, normalize=True)
# score_oos_xgb = mask_scores_by_price(score_oos_xgb, label_tag=LABEL_TAG)
# score_oos_tf = mask_scores_by_price(score_oos_tf, label_tag=LABEL_TAG)
# score_oos_mlp = mask_scores_by_price(score_oos_mlp, label_tag=LABEL_TAG)
#
# score_oos_ens = ensemble_scores(
#     {"xgb": score_oos_xgb, "transformer": score_oos_tf, "mlp": score_oos_mlp},
#     selected_models=selected,
#     label_tag=f"{LABEL_TAG}_oos",
#     save=True,
#     output_dir=config.SCORE_OUTPUT_DIR
# )
#
# run_quick_backtest(
#     score_df=score_oos_ens,
#     label_df=label_df,
#     n_groups=20,
#     output_dir=config.BT_RESULT_DIR / "oos" / "ensemble",
#     start_date=config.OOS_START,
#     top_ks=(20, 50, 100),
#     tail_ks=(20, 50, 100),
#     run_ic=True,
#     title=f"ENSEMBLE | {LABEL_TAG} | OOS"
# )
